# Teen Mental Health — Exploratory Data Analysis
**Target:** `depression_label` (binario: 0 = no depresión, 1 = depresión)

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

df = pd.read_csv('../data/Teen_Mental_Health_Dataset.csv')
print(df.shape)
df.head()

## 1. Inspección inicial

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
# Nulos y duplicados
print('Nulos por columna:')
print(df.isnull().sum())
print(f'\nDuplicados: {df.duplicated().sum()}')

## 2. Balance de clases (target)

In [ ]:
counts = df['depression_label'].value_counts()
pcts   = df['depression_label'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['No depresión (0)', 'Depresión (1)'], counts.values,
              color=['steelblue', 'salmon'], edgecolor='white')
for bar, pct in zip(bars, pcts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{pct:.1f}%', ha='center', fontsize=11)
ax.set_title('Balance de clases')
ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

## 3. Distribución de variables numéricas

In [ ]:
num_cols = ['age', 'daily_social_media_hours', 'sleep_hours',
            'screen_time_before_sleep', 'academic_performance',
            'physical_activity', 'stress_level', 'anxiety_level', 'addiction_level']

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for label, color in zip([0, 1], ['steelblue', 'salmon']):
        axes[i].hist(df[df['depression_label'] == label][col],
                     bins=20, alpha=0.6, color=color,
                     label=f'label={label}', density=True)
    axes[i].set_title(col, fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle('Distribución de variables numéricas por clase', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Variables categóricas vs. target

In [ ]:
cat_cols = ['gender', 'platform_usage', 'social_interaction_level']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, cat_cols):
    ct = df.groupby(col)['depression_label'].mean().sort_values(ascending=False)
    ct.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Tasa depresión por {col}', fontsize=10)
    ax.set_ylabel('P(depression=1)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylim(0, 1)

plt.suptitle('Tasa de depresión por variable categórica', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Matriz de correlación (variables numéricas)

In [ ]:
corr = df[num_cols + ['depression_label']].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Matriz de correlación', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Boxplots: variables numéricas vs. target

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='depression_label', y=col,
                palette={'0': 'steelblue', '1': 'salmon'}, ax=axes[i])
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('depression_label')

plt.suptitle('Boxplots por clase', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7. Correlación con el target (ranking)

In [ ]:
target_corr = df[num_cols + ['depression_label']].corr()['depression_label'].drop('depression_label').sort_values()

colors = ['salmon' if v > 0 else 'steelblue' for v in target_corr]
fig, ax = plt.subplots(figsize=(7, 5))
target_corr.plot(kind='barh', color=colors, edgecolor='white', ax=ax)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlación de Pearson con depression_label', fontsize=12)
ax.set_xlabel('Correlación')
plt.tight_layout()
plt.show()

print(target_corr.sort_values(ascending=False).to_string())